In [41]:
# Instalando as ferramentas necessárias
!pip install chromadb pypdf gradio huggingface_hub -q

In [43]:
# Importando bibliotecas
import chromadb
import gradio
import pypdf
from huggingface_hub import InferenceClient

In [44]:
from kaggle_secrets import UserSecretsClient
from huggingface_hub import InferenceClient

# Buscando o Token nos Secrets do Kaggle
user_secrets = UserSecretsClient()
token_hf = user_secrets.get_secret("HUGGING_FACE_API_KEY")

# Definindo o modelo e o cliente
modelo = "openai/gpt-oss-120b"
client = InferenceClient(model=modelo, token=token_hf)

print("✅ Modelo conectado!")

✅ Modelo conectado!


In [46]:
from pypdf import PdfReader
import textwrap

print("1. Lendo o Manual da GoodWe ...")
leitor = PdfReader("/kaggle/input/datasets/giovanesalazar/manual-goodwe/GW_HCA-G2_User-Manual-PT.pdf")

texto_completo = ""
for pagina in leitor.pages:
    texto_completo += pagina.extract_text() + "\n"

print("2. Fatiando o texto em chunks...")
chunks_contrato = textwrap.wrap(texto_completo, width=1000)
ids_chunks = [f"pedaco_{i}" for i in range(len(chunks_contrato))]

print(f"✅ Sucesso! {len(chunks_contrato)} blocos gerados.")

1. Lendo o Manual da GoodWe ...
2. Fatiando o texto em chunks...
✅ Sucesso! 71 blocos gerados.


In [47]:
import chromadb

cliente_db = chromadb.Client()
colecao = cliente_db.get_or_create_collection(name="base_goodwe")

print("Vetorizando e salvando no banco...")
colecao.add(documents=chunks_contrato, ids=ids_chunks)

print("✅ Banco de dados pronto!")

Vetorizando e salvando no banco...
✅ Banco de dados pronto!


In [57]:
def chatbot_goodwe(pergunta, historico):
    
    # 1. BUSCA: O ChromaDB encontra o trecho relevante
    busca = colecao.query(query_texts=[pergunta], n_results=5)
    contexto = busca['documents'][0][0] + '\n' + busca['documents'][0][1] + '\n' + busca['documents'][0][2]
    
    # 2. MENSAGENS: Estruturando os papéis (System e User)
    # É aqui que o Chat Completion se diferencia!
    mensagens = [
        {
            "role": "system", 
            "content": f"""Você é um assistente virtual especializado na solução GoodWe ChargeGrid Intelligence. Responda apenas com base no documento fornecido. (resuma a resposta e se o texto não tiver relação com a pergunta, diga que não há dados no documento para amparar a resposta):
            [CONTEXTO]
            {contexto}
            [/CONTEXTO]"""
        },
        {
            "role": "user", 
            "content": pergunta
        }
    ]
    
    # 3. GERAÇÃO: Chamando o modelo
    # Note como a sintaxe é limpa e organizada
    resposta_objeto = client.chat_completion(
        messages=mensagens,
        max_tokens=1000,
        temperature=0.1
    )
    
    # Extraindo apenas o texto da resposta final
    return resposta_objeto.choices[0].message.content

print("✅ Motor RAG configurado com Chat Completion!")

✅ Motor RAG configurado com Chat Completion!


In [62]:
import gradio as gr

def chatbot_goodwe(pergunta, historico):

    busca = colecao.query(
        query_texts=[pergunta],
        n_results=3
    )

    contexto = "\n".join(busca['documents'][0])

    mensagens = [
        {
            "role": "system",
            "content": f"""
Você é um assistente virtual especializado na solução GoodWe ChargeGrid Intelligence.

Seu objetivo é:
- responder perguntas sobre carregamento de veículos elétricos
- explicar métricas de eficiência energética
- auxiliar operadores de estações de carregamento
- responder apenas dentro do contexto do documento fornecido

Regras:
- não invente informações
- se não souber a resposta, diga claramente
- mantenha respostas técnicas e objetivas

[CONTEXTO]
{contexto}
[/CONTEXTO]
"""
        },
        {
            "role": "user",
            "content": pergunta
        }
    ]

    resposta = client.chat_completion(
        messages=mensagens,
        temperature=0.1
    )

    return resposta.choices[0].message.content


interface = gr.ChatInterface(
    fn=chatbot_goodwe,

    title="GoodWe EV Assistant",

    description="""
Chatbot contextualizado para o EV Challenge 2026.
""",

    theme="soft",

    examples=[
        "Como fazer Download e Instalação do Aplicativo?",
        "Como Desmantar o carregador?",          
        "Quais são as Funcionalidades?",
        "Como Desligar o carregador?",
        "Sobre a Conexão elétrica, quais são as precauções de segurança?",        
        
    ]
)

interface.launch()

/usr/local/lib/python3.12/dist-packages/gradio/chat_interface.py:347: UserWarning: The 'tuples' format for chatbot messages is deprecated and will be removed in a future version of Gradio. Please set type='messages' instead, which uses openai-style 'role' and 'content' keys.
  self.chatbot = Chatbot(


* Running on local URL:  http://127.0.0.1:7879
It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

* Running on public URL: https://257da900542e334687.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
